# Solution

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/finite_element_method/solution.ipynb)

We now arrive at a matrix equation for the unknowns $u_i$, which is merely a system of equations as we've spoken about beforehand.

### Linear PDE

If the PDE is linear, we can solve it with a linear solver which is sparse due to the small support requirement of the elements. The matrix will not in general be banded however, due to the irregular connectivity of the mesh vertices.

In [ ]:
try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

## Example: 1D heat equation

Solve the 1D heat balance equation with $\lambda = 1$ for $x$ from $0$ to $10$ subject to $J(0) = 1$ and $T(10) = 0$. Initial condition is $T(x, t = 0) = 0$.

Let $h = x_2 - x_1$. The element mass matrix is:

$$M_{ij} = \int_{x_1}^{x_2} \phi_i(x) \phi_j(x) \, \mathrm{d}x$$

with

$$\begin{aligned}
\phi_1(x) &= \frac{x_2 - x}{x_2 - x_1} \\
\phi_2(x) &= \frac{x - x_1}{x_2 - x_1}
\end{aligned}$$

which gives:

$$\mathbf{M}_e = \frac{h}{6} \begin{bmatrix}
2 & 1 \\
1 & 2
\end{bmatrix}$$

The element stiffness matrix needs

$$\begin{aligned}
\nabla \phi_1(x) &= \frac{-1}{x_2 - x_1} \\
\nabla \phi_2(x) &= \frac{1}{x_2 - x_1}
\end{aligned}$$

and

$$\begin{aligned}
K_{ij} &= \lambda \int_{x_1}^{x_2} \nabla \phi_i(x) \cdot \nabla \phi_j(x) \, \mathrm{d}x \\
\mathbf{K}_e &= \frac{\lambda}{h} \begin{bmatrix}
1 & -1 \\
-1 & 1
\end{bmatrix}
\end{aligned}$$

These *blocks* belong to each element, so we have to build the full matrix by *summing them*. Note this implies an overlap at common vertices between elements!

```{literalinclude} ../../executable_engineering/executable_engineering/fem_module.py
:language: python
:start-at: def fem_heat_1d_linear
:end-before: def fem_heat_1d_nonlinear_lagged
```

In [ ]:
exe.display_source(exe.fem_heat_1d_linear)

x_nodes, T, M, K = exe.fem_heat_1d_linear(t_final=10.0)
exe.plot_matrix_sparsity(M, title="Sparsity Pattern of Mass Matrix M").show()
exe.plot_fem_heat_1d(mode="linear").show()

## Nonlinear case

Note that in the linear example above we built the $\mathbf{M}$ and $\mathbf{K}$ matrices once, outside the time loop, and reused them for each time step! Even if we had adaptive time stepping, the matrices wouldn't change!

If the PDE is nonlinear, we simply have to use a root finder. Note that in this case we are looking for the parameters $u_i$ directly — the shape functions are unaffected and therefore only the coefficients of the matrix *blocks* are altered. This brings up the concept of *sparsity patterns* which can help reduce the overhead of sparse matrix representations.

Since modern finite element software needs to construct the weak form, it is usually equipped with symbolic logic capabilities and therefore able to calculate the Jacobian for use in the root-finding algorithm.

## Example: 1D nonlinear heat equation

Repeat the above exercise with $\lambda = 1 + T / 10$.

```{literalinclude} ../../executable_engineering/executable_engineering/fem_module.py
:language: python
:start-at: def fem_heat_1d_nonlinear_lagged
:end-before: def fem_heat_1d_nonlinear_root
```

In [ ]:
exe.display_source(exe.fem_heat_1d_nonlinear_lagged)
exe.plot_fem_heat_1d(mode="nonlinear_lagged").show()

## Nonlinear solve using a root finder

Hmmmm... something is funny in that code! Notice that above, $\lambda(T)$ was evaluated explicitly at the *previous* time step (and $\mathbf{M}$ was reassembled every step) rather than solving the coupled nonlinear equation implicitly.

Let's solve the nonlinear system properly at each time step using a root finder:

```{literalinclude} ../../executable_engineering/executable_engineering/fem_module.py
:language: python
:start-at: def fem_heat_1d_nonlinear_root
:end-before: def plot_matrix_sparsity
```

In [ ]:
exe.display_source(exe.fem_heat_1d_nonlinear_root)
exe.plot_fem_heat_1d(mode="nonlinear_root").show()